# Pilot V2 - Class-Based Telecom Tower Fragility

This notebook moves from one tower to a small class of towers.

Kid version: instead of testing one toy tower, we create a small family of similar telecom towers. Some are slightly taller, some have larger antenna areas, and some have different widths. Then we blow wind at them from 0, 22.5, and 45 degrees and fit one fragility curve for each wind direction.

This is still a pilot surrogate model. It is meant to prepare the workflow for a future Wang-style class-based framework.

## 1. Imports and Paths

The notebook uses only common scientific Python libraries. The output path is repository-relative so it works after downloading from GitHub.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

CACHE_DIR = REPO_ROOT / 'outputs' / '_cache'
MPL_CACHE_DIR = REPO_ROOT / 'outputs' / '_matplotlib_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_DIR))
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 130
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RANDOM_SEED = 123
rng = np.random.default_rng(RANDOM_SEED)

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'notebook_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')
print(f'Notebook outputs: {OUTPUT_DIR}')

## 2. Class-Based Pilot Assumptions

This notebook keeps the same wind-speed stripes as V1, but now it models many towers and three wind directions. The tower ranges are pilot values used to create a synthetic class, not final calibrated literature values.

In [ ]:
wind_speed_stripes_mps = np.round(np.arange(20.0, 50.0 + 0.1, 2.5), 2)

assumptions = {
    'project_name': 'Notebook Pilot V2 class-based telecom tower fragility under wind',
    'tower_class': 'self-supporting steel lattice telecommunication tower',
    'hazard': 'wind only',
    'intensity_measure': '10-minute mean wind speed in m/s',
    'wind_speed_stripes_mps': wind_speed_stripes_mps.tolist(),
    'wind_directions_deg': [0.0, 22.5, 45.0],
    'class_inventory_size': 80,
    'damage_state': 'collapse only',
    'synthetic_class_ranges': {
        'height_m': [44.0, 52.0],
        'base_width_m': [5.0, 7.5],
        'top_width_m': [1.6, 3.0],
        'lattice_solidity_ratio': [0.16, 0.28],
        'antenna_area_m2': [6.0, 18.0],
        'tower_drag_coefficient': [1.8, 2.1],
        'antenna_drag_coefficient': [1.4, 1.8],
    },
    'uncertainty': {
        'wind_load_multiplier_cov': 0.18,
        'antenna_load_multiplier_cov': 0.12,
        'capacity_multiplier_cov': 0.15,
        'baseline_capacity_factor_cov': 0.10,
    },
}

assumptions

## 3. Helper Functions

These functions create random but reasonable synthetic towers, compute simple wind loads, check collapse, and fit lognormal fragility curves. The function names are intentionally readable so this can later be swapped with structural-analysis code.

In [ ]:
def lognormal_mu_sigma_from_mean_cov(mean_value, cov_value):
    '''Convert mean and COV into lognormal log-space parameters.'''
    sigma = np.sqrt(np.log(1.0 + cov_value**2))
    mu = np.log(mean_value) - 0.5 * sigma**2
    return mu, sigma


def sample_lognormal(mean_value, cov_value, size):
    '''Sample positive lognormal random values.'''
    mu, sigma = lognormal_mu_sigma_from_mean_cov(mean_value, cov_value)
    return rng.lognormal(mean=mu, sigma=sigma, size=size)


def generate_synthetic_tower_class(assumptions_dict):
    '''Generate a synthetic telecom tower class inventory.'''
    ranges = assumptions_dict['synthetic_class_ranges']
    n_towers = assumptions_dict['class_inventory_size']

    height_m = rng.uniform(*ranges['height_m'], size=n_towers)
    base_width_m = rng.uniform(*ranges['base_width_m'], size=n_towers)
    top_width_m = rng.uniform(*ranges['top_width_m'], size=n_towers)
    solidity = rng.uniform(*ranges['lattice_solidity_ratio'], size=n_towers)
    antenna_area_m2 = rng.uniform(*ranges['antenna_area_m2'], size=n_towers)
    tower_cd = rng.uniform(*ranges['tower_drag_coefficient'], size=n_towers)
    antenna_cd = rng.uniform(*ranges['antenna_drag_coefficient'], size=n_towers)
    capacity_factor = sample_lognormal(1.0, assumptions_dict['uncertainty']['baseline_capacity_factor_cov'], n_towers)

    inventory_df = pd.DataFrame({
        'tower_id': np.arange(1, n_towers + 1),
        'height_m': height_m,
        'base_width_m': base_width_m,
        'top_width_m': top_width_m,
        'mean_face_width_m': 0.5 * (base_width_m + top_width_m),
        'lattice_solidity_ratio': solidity,
        'antenna_area_m2': antenna_area_m2,
        'tower_drag_coefficient': tower_cd,
        'antenna_drag_coefficient': antenna_cd,
        'baseline_capacity_factor': capacity_factor,
    })

    return inventory_df


def direction_factor(direction_deg):
    '''Simple wind-direction amplification for a square lattice tower.'''
    return 1.0 + 0.15 * np.sin(np.deg2rad(2.0 * direction_deg)) ** 2


def simulate_tower_response(tower_row, wind_speed_mps, direction_deg):
    '''Simulate one tower under one wind speed and direction.'''
    wind_multiplier = sample_lognormal(1.0, assumptions['uncertainty']['wind_load_multiplier_cov'], 1)[0]
    antenna_multiplier = sample_lognormal(1.0, assumptions['uncertainty']['antenna_load_multiplier_cov'], 1)[0]
    capacity_multiplier = sample_lognormal(1.0, assumptions['uncertainty']['capacity_multiplier_cov'], 1)[0]

    q_n_per_m2 = 0.613 * wind_speed_mps**2
    direction_amp = direction_factor(direction_deg)

    tower_area_m2 = tower_row['height_m'] * tower_row['mean_face_width_m'] * tower_row['lattice_solidity_ratio'] * direction_amp
    antenna_area_m2 = tower_row['antenna_area_m2'] * (1.0 + 0.10 * (direction_amp - 1.0) / 0.15)

    tower_force_kN = q_n_per_m2 * tower_area_m2 * tower_row['tower_drag_coefficient'] * wind_multiplier / 1000.0
    antenna_force_kN = q_n_per_m2 * antenna_area_m2 * tower_row['antenna_drag_coefficient'] * wind_multiplier * antenna_multiplier / 1000.0
    total_force_kN = tower_force_kN + antenna_force_kN
    overturning_moment_kNm = total_force_kN * 0.60 * tower_row['height_m']

    total_capacity_factor = tower_row['baseline_capacity_factor'] * capacity_multiplier
    leg_capacity_kNm = 2600.0 * (tower_row['base_width_m'] / 6.0) ** 1.20 * (48.0 / tower_row['height_m']) ** 0.30 * total_capacity_factor
    brace_capacity_kN = 95.0 * (tower_row['mean_face_width_m'] / 4.0) * (48.0 / tower_row['height_m']) ** 0.20 * total_capacity_factor
    antenna_capacity_kN = 18.0 * (48.0 / tower_row['height_m']) ** 0.10 * total_capacity_factor

    leg_utilization = overturning_moment_kNm / leg_capacity_kNm
    brace_utilization = total_force_kN / brace_capacity_kN
    antenna_utilization = antenna_force_kN / antenna_capacity_kN
    global_index = 0.60 * leg_utilization + 0.25 * brace_utilization + 0.15 * antenna_utilization

    component_failures = int(leg_utilization > 1.0) + int(brace_utilization > 1.0) + int(antenna_utilization > 1.0)
    collapse = (leg_utilization > 1.05) or (global_index > 1.0) or (component_failures >= 2)

    return {
        'tower_force_kN': float(tower_force_kN),
        'antenna_force_kN': float(antenna_force_kN),
        'total_force_kN': float(total_force_kN),
        'overturning_moment_kNm': float(overturning_moment_kNm),
        'leg_utilization': float(leg_utilization),
        'brace_utilization': float(brace_utilization),
        'antenna_utilization': float(antenna_utilization),
        'global_collapse_index': float(global_index),
        'collapse': bool(collapse),
    }

## 4. Create the Synthetic Tower Class

This table is the fake class inventory for the pilot. Later, this would be replaced by real tower inventory data or literature-calibrated class parameters.

In [ ]:
tower_inventory_df = generate_synthetic_tower_class(assumptions)
tower_inventory_df.head()

## 5. Run Class-Based Stripe Analysis

For every wind direction and wind speed, we test all 80 synthetic towers. The observed collapse probability is the fraction of towers that collapsed.

In [ ]:
simulation_records = []
stripe_records = []

for direction_deg in assumptions['wind_directions_deg']:
    for wind_speed_mps in assumptions['wind_speed_stripes_mps']:
        collapse_flags = []
        global_indices = []

        for _, tower_row in tower_inventory_df.iterrows():
            response = simulate_tower_response(tower_row, wind_speed_mps, direction_deg)
            collapse_flags.append(int(response['collapse']))
            global_indices.append(response['global_collapse_index'])

            simulation_records.append({
                'tower_id': int(tower_row['tower_id']),
                'direction_deg': float(direction_deg),
                'wind_speed_mps': float(wind_speed_mps),
                'height_m': float(tower_row['height_m']),
                'antenna_area_m2': float(tower_row['antenna_area_m2']),
                **response,
            })

        stripe_records.append({
            'direction_deg': float(direction_deg),
            'wind_speed_mps': float(wind_speed_mps),
            'simulations_in_stripe': len(collapse_flags),
            'failure_count': int(np.sum(collapse_flags)),
            'observed_collapse_probability': float(np.mean(collapse_flags)),
            'mean_global_collapse_index': float(np.mean(global_indices)),
        })

simulation_results_df = pd.DataFrame(simulation_records)
stripe_results_df = pd.DataFrame(stripe_records)
stripe_results_df.head(10)

## 6. Fit Fragility Curves by Wind Direction

This is the same lognormal curve idea as V1, but now we fit a separate curve for 0, 22.5, and 45 degrees.

In [ ]:
def lognormal_fragility_probability(wind_speed_mps, theta, beta):
    '''Compute collapse probability from a lognormal fragility curve.'''
    wind_speed_mps = np.asarray(wind_speed_mps, dtype=float)
    probability = norm.cdf((np.log(wind_speed_mps) - np.log(theta)) / beta)
    return np.clip(probability, 1e-10, 1.0 - 1e-10)


def negative_log_likelihood(log_parameters, wind_speeds, failures, totals):
    '''Binomial negative log-likelihood for fitting stripe data.'''
    theta = np.exp(log_parameters[0])
    beta = np.exp(log_parameters[1])
    p_fail = lognormal_fragility_probability(wind_speeds, theta, beta)
    log_likelihood = failures * np.log(p_fail) + (totals - failures) * np.log(1.0 - p_fail)
    return -float(np.sum(log_likelihood))


fit_records = []

for direction_deg in assumptions['wind_directions_deg']:
    direction_df = stripe_results_df[stripe_results_df['direction_deg'] == direction_deg]
    wind_speeds = direction_df['wind_speed_mps'].to_numpy(dtype=float)
    failures = direction_df['failure_count'].to_numpy(dtype=float)
    totals = direction_df['simulations_in_stripe'].to_numpy(dtype=float)

    fit_result = minimize(
        negative_log_likelihood,
        x0=np.log([38.0, 0.18]),
        args=(wind_speeds, failures, totals),
        method='L-BFGS-B',
        bounds=[(np.log(1.0), np.log(200.0)), (np.log(0.03), np.log(2.0))],
    )

    if not fit_result.success:
        raise RuntimeError(f'Fit failed for direction {direction_deg}: {fit_result.message}')

    fit_records.append({
        'direction_deg': float(direction_deg),
        'theta_mps': float(np.exp(fit_result.x[0])),
        'beta': float(np.exp(fit_result.x[1])),
        'negative_log_likelihood': float(fit_result.fun),
    })

fragility_fit_df = pd.DataFrame(fit_records)
fragility_fit_df

## 7. Plot Direction-Specific Fragility Curves

Each color is one wind direction. The dots are simulated class results. The lines are fitted fragility curves.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
smooth_wind_speeds = np.linspace(20.0, 50.0, 300)
colors = {0.0: 'tab:blue', 22.5: 'tab:orange', 45.0: 'tab:red'}

for _, fit_row in fragility_fit_df.iterrows():
    direction_deg = fit_row['direction_deg']
    direction_df = stripe_results_df[stripe_results_df['direction_deg'] == direction_deg]

    ax.scatter(
        direction_df['wind_speed_mps'],
        direction_df['observed_collapse_probability'],
        color=colors[direction_deg],
        s=55,
        alpha=0.85,
        label=f'Observed {direction_deg:g} deg',
    )
    ax.plot(
        smooth_wind_speeds,
        lognormal_fragility_probability(smooth_wind_speeds, fit_row['theta_mps'], fit_row['beta']),
        color=colors[direction_deg],
        linewidth=2.3,
        label=f'Fitted {direction_deg:g} deg',
    )

ax.set_xlabel('10-minute mean wind speed, V (m/s)')
ax.set_ylabel('Probability of collapse')
ax.set_title('Notebook Pilot V2 class-based telecom tower fragility')
ax.set_ylim(-0.02, 1.02)
ax.legend(ncol=2, fontsize=9)
ax.grid(alpha=0.3)
plt.show()

## 8. Save Notebook Outputs

The notebook saves the synthetic inventory, simulation results, stripe table, and fitted fragility summary. These outputs are useful for checking or reusing the pilot results.

In [ ]:
with (OUTPUT_DIR / 'assumptions.json').open('w', encoding='utf-8') as file:
    json.dump(assumptions, file, indent=4)

tower_inventory_df.to_csv(OUTPUT_DIR / 'synthetic_tower_class_inventory.csv', index=False)
simulation_results_df.to_csv(OUTPUT_DIR / 'simulation_results.csv', index=False)
stripe_results_df.to_csv(OUTPUT_DIR / 'stripe_results.csv', index=False)
fragility_fit_df.to_csv(OUTPUT_DIR / 'fragility_fit_summary.csv', index=False)

print(f'Saved notebook V2 outputs to: {OUTPUT_DIR}')
fragility_fit_df